## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

In [ ]:
# The imports

import os
import google.auth
import google.auth.transport.requests
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, OpenAIChatCompletionsModel, set_tracing_disabled


In [ ]:
# Load env, set up Vertex AI client, and disable OpenAI tracing
load_dotenv(override=True)

project_id = os.getenv("GCP_PROJECT")
location   = os.getenv("GCP_LOCATION", "us-central1")
model_name = os.getenv("MODEL_NAME", "google/gemini-2.5-pro")

def _get_vertex_async_client():
    creds, _ = google.auth.default()
    req = google.auth.transport.requests.Request()
    creds.refresh(req)
    return AsyncOpenAI(
        base_url=(
            f"https://{location}-aiplatform.googleapis.com/v1/projects/{project_id}"
            f"/locations/{location}/endpoints/openapi"
        ),
        api_key=creds.token,
    )

vertex_async_client = _get_vertex_async_client()
vertex_model = OpenAIChatCompletionsModel(model=model_name, openai_client=vertex_async_client)

# Disable OpenAI platform tracing — not using the OpenAI API
set_tracing_disabled(True)

print(f"Project : {project_id}")
print(f"Location: {location}")
print(f"Model   : {model_name}")


In [ ]:
# Make an agent with name, instructions, model (Vertex AI Gemini)
agent = Agent(name="Jokester", instructions="You are a joke teller", model=vertex_model)


In [ ]:
# Run the joke with Runner.run(agent, prompt) then print final_output
# (trace() is a no-op since set_tracing_disabled(True) was called above)
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
    print(result.final_output)


## Now go and look at the trace

https://platform.openai.com/traces